# SAiDL AFT + RL Bonus Kaggle Runner (Repo-First + Hydra + W&B)

This notebook runs only the extended experiments now needed after the deadline extension:

- Core ML AFT bonus variants.
- RL bonus positional-encoding ablation.
- RL combined POMDP.
- RL Algorithm Distillation from TD3 source-policy histories.
- RL xLSTM policy-backbone runs.

It keeps the repo-first workflow: Kaggle clones or restores the checked-in repository, then executes Hydra commands from the repo folders. Long commands are run through checkpointed `run_block(...)` blocks, so completed commands are skipped on rerun and TD3/AFT checkpoints are resumed when available.

---
## Before Running
1. In Kaggle notebook settings: enable **GPU** and **Internet**.
2. In Kaggle Add-ons -> Secrets, add `WANDB_API_KEY`.
3. If the repo is private, add `GITHUB_TOKEN` as a Kaggle Secret.
4. For Algorithm Distillation, attach a prior TD3 output dataset containing a source checkpoint, or set `RL_AD_SOURCE_CKPT` explicitly.


In [ ]:
# 1) Install dependencies in the active notebook interpreter
import sys
import subprocess

REQ = [
    "torch==2.4.1",
    "hydra-core",
    "omegaconf",
    "datasets",
    "transformers",
    "wandb",
    "gymnasium",
    "matplotlib",
    "numpy",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *REQ])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gymnasium[mujoco]", "mujoco"])
# CUDA pre-flight check: catch PyTorch / GPU compute-capability mismatch early
import torch as _torch
if _torch.cuda.is_available():
    try:
        _torch.zeros(1, device="cuda")
        print(f"CUDA OK: {_torch.cuda.get_device_name(0)}, CC {_torch.cuda.get_device_capability()}")
    except Exception as _e:
        raise RuntimeError(
            f"CUDA available but kernels fail. PyTorch/GPU mismatch (likely P100 + PyTorch>=2.5): {_e}"
        )
else:
    print("WARNING: No CUDA device detected. Training will use CPU.")
del _torch
print(f"Dependency install complete via interpreter: {sys.executable}")

In [ ]:
# 2) Authenticate W&B and load optional GitHub token from Kaggle Secrets
import os
import wandb

def _get_secret(name: str) -> str:
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        secret_client = UserSecretsClient()
        value = (secret_client.get_secret(name) or "").strip()
        if value:
            os.environ[name] = value
        return value
    except Exception:
        return ""

wandb_key = _get_secret("WANDB_API_KEY")
if wandb_key:
    wandb.login(key=wandb_key)
    print("W&B login successful.")
else:
    print("WANDB_API_KEY not found in env/secrets. You can run with logging.wandb.enable=false.")

github_token = _get_secret("GITHUB_TOKEN")
if github_token:
    print("GITHUB_TOKEN loaded from env/secrets.")
else:
    print("GITHUB_TOKEN not found. Private GitHub clone will fail unless dataset fallback is attached.")

In [ ]:
# 3) Bring your existing repo into /kaggle/working
from pathlib import Path
import os
import glob
import shutil
import zipfile
import subprocess
from urllib.parse import quote
from urllib.request import Request, urlopen

REPO_URL = os.environ.get("SAIDL_REPO_URL", "https://github.com/Krish-Sachdev-7/SAiDL-Summer-Assignment-2026.git")
REPO_REF = os.environ.get("SAIDL_REPO_REF", "main")
REPO_DIR = Path("/kaggle/working/SAiDL_Assignment")
ZIP_HINT = os.environ.get("SAIDL_ZIP_PATH", "")

def get_secret(name: str) -> str:
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        secret_client = UserSecretsClient()
        value = (secret_client.get_secret(name) or "").strip()
        if value:
            os.environ[name] = value
        return value
    except Exception:
        return ""

def extract_zip(zip_path: Path, dest_dir: Path):
    print(f"Extracting {zip_path} -> {dest_dir}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest_dir)

def find_zip_fallback():
    candidates = []
    for pattern in ["/kaggle/input/**/*.zip", "/kaggle/working/**/*.zip"]:
        candidates.extend(glob.glob(pattern, recursive=True))
    preferred = [p for p in candidates if "saidl" in p.lower() and "assignment" in p.lower()]
    if preferred:
        return Path(preferred[0])
    return Path(candidates[0]) if candidates else None

def find_dataset_folder_fallback():
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None
    candidates = []
    for d in input_root.iterdir():
        if d.is_dir():
            if (d / "core_ml").exists() and (d / "rl").exists():
                candidates.append(d)
                continue
            nested = [p for p in d.iterdir() if p.is_dir() and (p / "core_ml").exists() and (p / "rl").exists()]
            candidates.extend(nested)
    preferred = [p for p in candidates if "saidl" in p.name.lower()]
    if preferred:
        return preferred[0]
    return candidates[0] if candidates else None

def github_archive_fallback(repo_url: str, ref: str, token: str, out_dir: Path) -> bool:
    if not token or not repo_url.startswith("https://github.com/"):
        return False

    repo_path = repo_url.replace("https://github.com/", "").strip()
    if repo_path.endswith(".git"):
        repo_path = repo_path[:-4]
    if "/" not in repo_path:
        return False

    api_url = f"https://api.github.com/repos/{repo_path}/zipball/{ref}"
    tmp_zip = out_dir / "github_repo_fallback.zip"
    print(f"Trying GitHub API archive fallback for ref {ref}...")
    try:
        req = Request(
            api_url,
            headers={
                "Authorization": f"Bearer {token}",
                "Accept": "application/vnd.github+json",
            },
        )
        with urlopen(req) as response, open(tmp_zip, "wb") as out:
            out.write(response.read())
        extract_zip(tmp_zip, out_dir)
        return True
    except Exception as exc:
        print(f"GitHub API archive fallback failed: {exc}")
        return False

def clone_repo():
    token = get_secret("GITHUB_TOKEN")
    print(f"GITHUB_TOKEN available to Cell 3: {'yes' if bool(token) else 'no'}")
    clone_cmd = ["git", "clone", "--branch", REPO_REF, "--single-branch"]
    if token and REPO_URL.startswith("https://github.com/"):
        enc_token = quote(token, safe="")
        auth_url = REPO_URL.replace("https://", f"https://x-access-token:{enc_token}@")
        print(f"Trying authenticated git clone of ref {REPO_REF} via GITHUB_TOKEN...")
        subprocess.run([*clone_cmd, auth_url, str(REPO_DIR)], check=True)
        return
    print(f"Trying git clone of ref {REPO_REF} from: {REPO_URL}")
    subprocess.run([*clone_cmd, REPO_URL, str(REPO_DIR)], check=True)

def ensure_conv_none_kernel_size(repo_dir: Path) -> None:
    # Compatibility patch for older snapshots lacking conv.kernel_size in conv/none.yaml.
    none_cfg = repo_dir / "core_ml" / "conf" / "conv" / "none.yaml"
    if not none_cfg.exists():
        return
    text = none_cfg.read_text(encoding="utf-8")
    if "kernel_size:" in text:
        return
    patched = text.rstrip() + "\nkernel_size: 3\n"
    none_cfg.write_text(patched, encoding="utf-8")
    print("Applied compatibility patch: added kernel_size to core_ml/conf/conv/none.yaml")

if REPO_DIR.exists():
    print(f"Repo already present: {REPO_DIR}")
else:
    cloned = False
    try:
        clone_repo()
        cloned = True
        print("Clone succeeded.")
    except Exception as exc:
        print(f"Clone failed: {exc}")
        print("Trying GitHub archive and dataset fallbacks...")

    if not cloned:
        token = get_secret("GITHUB_TOKEN")
        if github_archive_fallback(REPO_URL, REPO_REF, token, Path("/kaggle/working")):
            print("GitHub archive fallback succeeded.")
        else:
            zip_path = Path(ZIP_HINT) if ZIP_HINT else find_zip_fallback()
            if zip_path is not None and zip_path.exists():
                extract_zip(zip_path, Path("/kaggle/working"))
            else:
                folder_src = find_dataset_folder_fallback()
                if folder_src is None:
                    raise FileNotFoundError(
                        "Could not clone repo and no fallback source found. "
                        "Add GITHUB_TOKEN in Kaggle secrets, or attach a Kaggle Dataset (zip/folder), "
                        "or set SAIDL_ZIP_PATH."
                    )
                print(f"Copying dataset folder: {folder_src} -> {REPO_DIR}")
                shutil.copytree(folder_src, REPO_DIR, dirs_exist_ok=True)

        if not REPO_DIR.exists():
            extracted_dirs = [p for p in Path("/kaggle/working").iterdir() if p.is_dir()]
            saidl_candidates = [p for p in extracted_dirs if "saidl" in p.name.lower()]
            if saidl_candidates:
                src = saidl_candidates[0]
                print(f"Using extracted folder: {src}")
                if src != REPO_DIR:
                    shutil.move(str(src), str(REPO_DIR))

if not REPO_DIR.exists():
    raise FileNotFoundError(f"Repo directory still missing after fallback attempts: {REPO_DIR}")

ensure_conv_none_kernel_size(REPO_DIR)

print(f"Using repo at: {REPO_DIR}")
print("Top-level files:")
for p in sorted(REPO_DIR.iterdir())[:30]:
    print(" -", p.name)


In [ ]:
# 4) Verify critical files exist
!ls -la /kaggle/working/SAiDL_Assignment
!ls -la /kaggle/working/SAiDL_Assignment/core_ml/src
!ls -la /kaggle/working/SAiDL_Assignment/rl/src

In [ ]:
# 5) Optional sanity checks
import os

RUN_SANITY = os.environ.get("RUN_SANITY", "0") == "1"
if RUN_SANITY:
    print("RUN_SANITY=1: sanity checks are enabled in the next cell.")
else:
    print("Skipping sanity checks. Set RUN_SANITY=1 to run short smoke commands before final runs.")


In [ ]:
# 5b) Optional RL sanity run (short)
import os
import sys
import subprocess
from pathlib import Path

if os.environ.get("RUN_SANITY", "0") == "1":
    rl_dir = Path("/kaggle/working/SAiDL_Assignment/rl")
    print(f"Running RL sanity in: {rl_dir}")
    subprocess.check_call([
        sys.executable,
        "src/train.py",
        "experiment.name=sanity_rl",
        "total_steps=2000",
        "eval_interval=1000",
        "eval_episodes=2",
        "logging.wandb.enable=false",
    ], cwd=str(rl_dir))
else:
    print("RUN_SANITY is disabled; skipping RL sanity run.")


# Stabilized RoPE Comparison Runner

This notebook version intentionally launches only one RL comparison run:

```bash
python src/train.py experiment.name=rl_bonus_pos_rope_stable_seed42 agent=td3_transformer env=hopper_hidden_vel +pos_encoding=rope
```

The run keeps the same environment, seed, Transformer actor, and RoPE positional encoding as the previous `rl_bonus_pos_rope_seed42` run, but uses the stability-focused source changes and conservative TD3 overrides:

- best-eval checkpoint saving and best/final metric logging,
- observation normalization when `agent.actor.obs_norm=true`,
- lower TD3 learning rate,
- lower exploration and target smoothing noise,
- slower target network updates.

Use this version to compare the stabilized rerun against the earlier unstable RoPE run.


# Fast Kaggle Workflow

1. Run setup cells through repo sync.
2. Keep `INCLUDE_CORE_BONUS=1` and `INCLUDE_RL_BONUS=1` unless intentionally disabling a block.
3. Attach previous TD3 output checkpoints for Algorithm Distillation, or set `RL_AD_SOURCE_CKPT`.
4. Run the AFT cell and RL bonus cell.
5. If interrupted, rerun the same cells. `run_block(...)` skips completed commands, TD3/AFT resumes from checkpoints, and AD training resumes from `ad_latest.pt`.


# Production Hardening: AFT + RL Bonus Runs

This section is designed for final bonus runs with minimal manual intervention.

Step-wise recovery layers:

- Command progress is checkpointed under `/kaggle/working/SAiDL_Assignment/.kaggle_runner_progress`.
- TD3 and AFT commands save model checkpoints at configured intervals.
- `run_block(...)` searches local and attached Kaggle inputs for matching checkpoints before launching each command.
- Algorithm Distillation writes `ad_latest.pt` during training and resumes from it on rerun.

If a command fails, fix the root cause and rerun the same cell. Previously completed commands are not repeated unless `FORCE_RERUN=True`.


In [ ]:

# Final-run helpers: resume support, fail-fast execution, and shared overrides
from pathlib import Path
import os
import sys
import shlex
import time
import subprocess
import importlib.util
import re
from typing import List, Optional, Tuple

REPO_ROOT = Path("/kaggle/working/SAiDL_Assignment")
CORE_DIR = REPO_ROOT / "core_ml"
RL_DIR = REPO_ROOT / "rl"
PROGRESS_DIR = REPO_ROOT / ".kaggle_runner_progress"
PROGRESS_DIR.mkdir(parents=True, exist_ok=True)

assert CORE_DIR.exists(), f"Missing Core ML directory: {CORE_DIR}"
assert RL_DIR.exists(), f"Missing RL directory: {RL_DIR}"

DRY_RUN = False
FORCE_RERUN = False
PYTHON_EXE = os.environ.get("PYTHON_EXE", sys.executable)

WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "").strip()
CORE_COMMON_OVERRIDES: List[str] = []
RL_COMMON_OVERRIDES: List[str] = ["checkpoint_interval=10000"]
if WANDB_ENTITY:
    CORE_COMMON_OVERRIDES.append(f"logging.wandb.entity={WANDB_ENTITY}")
    RL_COMMON_OVERRIDES.append(f"logging.wandb.entity={WANDB_ENTITY}")

def _safe_name(text: str) -> str:
    text = re.sub(r"[^a-zA-Z0-9_.-]+", "_", str(text)).strip("._-")
    return text or "run"

def ensure_runtime_packages() -> None:
    pkg_map = {
        "hydra": "hydra-core",
        "omegaconf": "omegaconf",
        "datasets": "datasets",
        "transformers": "transformers",
        "wandb": "wandb",
        "numpy": "numpy",
    }
    missing = [name for name in pkg_map if importlib.util.find_spec(name) is None]
    if not missing:
        print("Runtime package check passed.")
        return

    targets = [pkg_map[name] for name in missing]
    print(f"Installing missing runtime packages: {targets}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *targets])

def with_overrides(base_cmd: str, extra_overrides: List[str]) -> str:
    return base_cmd if not extra_overrides else base_cmd + " " + " ".join(extra_overrides)

def _progress_file(block_name: str) -> Path:
    safe_name = block_name.replace(" ", "_").replace("/", "_")
    return PROGRESS_DIR / f"{safe_name}.done.txt"

def _load_completed(block_name: str) -> set[str]:
    path = _progress_file(block_name)
    if not path.exists():
        return set()
    return set(line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip())

def _mark_completed(block_name: str, cmd: str) -> None:
    path = _progress_file(block_name)
    existing = _load_completed(block_name)
    if cmd in existing:
        return
    with path.open("a", encoding="utf-8") as f:
        f.write(cmd + "\n")

def _merge_progress_file(src: Path, dst: Path) -> None:
    src_lines = set(line.strip() for line in src.read_text(encoding="utf-8").splitlines() if line.strip())
    dst_lines = set()
    if dst.exists():
        dst_lines = set(line.strip() for line in dst.read_text(encoding="utf-8").splitlines() if line.strip())
    merged = sorted(dst_lines.union(src_lines))
    dst.write_text("\n".join(merged) + ("\n" if merged else ""), encoding="utf-8")

def restore_previous_kaggle_state() -> None:
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return
    restored = 0
    for prior_dir in input_root.rglob(".kaggle_runner_progress"):
        if not prior_dir.is_dir():
            continue
        for prior_file in prior_dir.glob("*.done.txt"):
            _merge_progress_file(prior_file, PROGRESS_DIR / prior_file.name)
            restored += 1
    if restored:
        print(f"Restored/merged {restored} progress file(s) from attached Kaggle inputs.")

def _cmd_value(cmd: str, key: str) -> Optional[str]:
    prefix = key + "="
    for token in shlex.split(cmd):
        if token.startswith(prefix):
            return token.split("=", 1)[1]
    return None

def _experiment_name_from_cmd(cmd: str) -> Optional[str]:
    return _cmd_value(cmd, "experiment.name")

def _target_max_steps_from_cmd(cmd: str) -> Optional[int]:
    for key in ["trainer.max_steps", "max_steps", "total_steps"]:
        value = _cmd_value(cmd, key)
        if value is not None:
            return int(value)
    return None

def _command_agent_type(cmd: str) -> Optional[str]:
    agent = _cmd_value(cmd, "agent")
    return {
        "td3_mlp": "mlp",
        "td3_transformer": "transformer",
        "td3_xlstm": "xlstm",
    }.get(agent)

def _command_context_length(cmd: str) -> Optional[int]:
    explicit = _cmd_value(cmd, "agent.actor.context_length")
    if explicit is not None:
        return int(explicit)
    agent_type = _command_agent_type(cmd)
    if agent_type in {"transformer", "xlstm"}:
        return 8
    if agent_type == "mlp":
        return 1
    return None

def _checkpoint_info(path: Path) -> dict:
    import torch
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(ck, dict):
        raise ValueError(f"Checkpoint is not a dict: {type(ck).__name__}")
    return ck

def _checkpoint_step_from_info(ck: dict) -> int:
    return int(ck.get("step", 0))

def _tensor_first_dim(value) -> Optional[int]:
    shape = getattr(value, "shape", None)
    if shape is None or len(shape) == 0:
        return None
    return int(shape[0])

def _checkpoint_matches_cmd(path: Path, ck: dict, experiment_name: str, cmd: str) -> bool:
    safe = _safe_name(experiment_name)
    normalized = str(path).replace("\\", "/")
    if experiment_name in normalized or safe in normalized:
        return True

    saved_experiment = ck.get("experiment_name")
    if saved_experiment:
        return str(saved_experiment) == experiment_name

    # Legacy RL checkpoints from earlier runner versions were saved as rl/ckpt_latest_rl.pt
    # without an experiment name. Only auto-match those when the command has an explicit
    # transformer context length that uniquely identifies the context-sweep partial run.
    if not normalized.endswith("/rl/ckpt_latest_rl.pt") and not normalized.endswith("/rl/final_rl_model.pt"):
        return False

    if _cmd_value(cmd, "agent.actor.context_length") is None:
        return False

    agent_type = _command_agent_type(cmd)
    if agent_type != "transformer":
        return False

    actor = ck.get("actor")
    if not isinstance(actor, dict):
        return False
    if "pos_emb.weight" not in actor or not any(str(k).startswith("blocks.") for k in actor):
        return False

    wanted_context = _command_context_length(cmd)
    saved_context = ck.get("context_length") or _tensor_first_dim(actor.get("pos_emb.weight"))
    return wanted_context is not None and saved_context == wanted_context

def _candidate_checkpoints(experiment_name: str, cmd: str) -> List[Path]:
    safe = _safe_name(experiment_name)
    candidates: List[Path] = []
    core_names = ["ckpt_latest.pt", "final_model.pt", "best_model.pt"]
    rl_names = ["ckpt_latest_rl.pt", "final_rl_model.pt"]

    def add(path: Path) -> None:
        if path.exists() and path not in candidates:
            candidates.append(path)

    for root in [CORE_DIR, RL_DIR, REPO_ROOT]:
        for name in core_names:
            add(root / "checkpoints" / safe / name)
        for name in rl_names:
            add(root / "checkpoints" / safe / name)

    search_roots = [CORE_DIR, RL_DIR]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        search_roots.append(input_root)

    for root in search_roots:
        if not root.exists():
            continue
        for name in core_names:
            for p in root.rglob(name):
                normalized = str(p).replace("\\", "/")
                if f"/checkpoints/{safe}/" in normalized:
                    add(p)
        for name in rl_names:
            for p in root.rglob(name):
                normalized = str(p).replace("\\", "/")
                if experiment_name in normalized or safe in normalized:
                    add(p)
                elif normalized.endswith(f"/rl/{name}"):
                    add(p)
    return candidates

def _find_resume_checkpoint(experiment_name: str, cmd: str) -> Tuple[Optional[Path], int]:
    best_path: Optional[Path] = None
    best_step = -1
    for path in _candidate_checkpoints(experiment_name, cmd):
        try:
            ck = _checkpoint_info(path)
            if not _checkpoint_matches_cmd(path, ck, experiment_name, cmd):
                continue
            step = _checkpoint_step_from_info(ck)
        except Exception as exc:
            print(f"Skipping unreadable checkpoint {path}: {exc}")
            continue
        if step > best_step:
            best_path = path
            best_step = step
    return best_path, best_step

def _with_resume_if_available(cmd: str, block_name: str) -> Optional[str]:
    experiment_name = _experiment_name_from_cmd(cmd)
    target_steps = _target_max_steps_from_cmd(cmd)
    if not experiment_name or target_steps is None:
        return cmd

    ckpt_path, ckpt_step = _find_resume_checkpoint(experiment_name, cmd)
    if ckpt_path is None:
        return cmd

    if "resume=" in cmd:
        return cmd

    if ckpt_step >= target_steps:
        print(
            f"Checkpoint already reached target for {experiment_name}: "
            f"step {ckpt_step} >= {target_steps}; rerunning finalization from checkpoint"
        )
        return cmd + f" +resume={shlex.quote(str(ckpt_path))}"

    print(f"Resuming {experiment_name} from {ckpt_path} at step {ckpt_step}/{target_steps}")
    return cmd + f" +resume={shlex.quote(str(ckpt_path))}"

def run_block(block_name: str, cwd: Path, commands: List[str]) -> None:
    ensure_runtime_packages()
    restore_previous_kaggle_state()

    completed = set() if FORCE_RERUN else _load_completed(block_name)
    pending = [cmd for cmd in commands if cmd not in completed]

    print(f"\n=== {block_name} ===")
    print(f"Working directory: {cwd}")
    print(f"Python executable: {PYTHON_EXE}")
    print(f"Total commands: {len(commands)}")
    print(f"Already completed: {len(commands) - len(pending)}")
    print(f"Pending now: {len(pending)}")

    if not pending:
        print("Nothing to run for this block.")
        return

    block_start = time.time()
    for idx, cmd in enumerate(pending, start=1):
        resumed_cmd = _with_resume_if_available(cmd, block_name)
        if resumed_cmd is None:
            continue

        exec_cmd = resumed_cmd.strip()
        if exec_cmd.startswith("python "):
            exec_cmd = f"{shlex.quote(PYTHON_EXE)} {exec_cmd[len('python '):]}"

        print(f"\n[{idx}/{len(pending)}] {exec_cmd}")
        if DRY_RUN:
            continue

        t0 = time.time()
        result = subprocess.run(["bash", "-lc", exec_cmd], cwd=str(cwd))
        dt = time.time() - t0
        print(f"Return code: {result.returncode} | Elapsed: {dt / 60.0:.2f} min")

        if result.returncode != 0:
            raise RuntimeError(
                f"Block '{block_name}' failed. Fix the issue, then rerun this cell to resume from the failure point."
            )
        _mark_completed(block_name, cmd)

    block_dt = time.time() - block_start
    print(f"\nCompleted block '{block_name}' in {block_dt / 60.0:.2f} minutes.")

restore_previous_kaggle_state()
print("Helper setup complete.")
print(f"Progress directory: {PROGRESS_DIR}")
print(f"Python executable: {PYTHON_EXE}")
print(f"DRY_RUN={DRY_RUN} FORCE_RERUN={FORCE_RERUN}")
if WANDB_ENTITY:
    print(f"Using W&B entity override: {WANDB_ENTITY}")
else:
    print("WANDB_ENTITY is not set. Project defaults will be used.")


In [ ]:
# Final-run controls for the single stabilized RoPE comparison
RL_TOTAL_STEPS = int(os.environ.get("RL_TOTAL_STEPS", "1000000"))
RL_EVAL_INTERVAL = int(os.environ.get("RL_EVAL_INTERVAL", "5000"))
RL_EVAL_EPISODES = int(os.environ.get("RL_EVAL_EPISODES", "10"))
RL_COMPARE_SEED = int(os.environ.get("RL_COMPARE_SEED", "42"))

INCLUDE_CORE_BONUS = False
INCLUDE_RL_BONUS = os.environ.get("INCLUDE_RL_BONUS", "1") == "1"
RUN_RLHF_WITH_MLP = False

print("Single-run stability comparison controls loaded:")
print(f"RL_TOTAL_STEPS={RL_TOTAL_STEPS} RL_EVAL_INTERVAL={RL_EVAL_INTERVAL} RL_EVAL_EPISODES={RL_EVAL_EPISODES}")
print(f"RL_COMPARE_SEED={RL_COMPARE_SEED}")
print(f"INCLUDE_CORE_BONUS={INCLUDE_CORE_BONUS} INCLUDE_RL_BONUS={INCLUDE_RL_BONUS}")
print("This version launches only rl_bonus_pos_rope_stable_seed{RL_COMPARE_SEED}.")


In [ ]:
# Core ML bonus suite intentionally disabled in this single-run RL stability notebook
print("Skipping Core ML AFT bonus suite in this notebook version.")


In [ ]:
# Required Core ML suite intentionally disabled
print("Skipping Core ML required suite in this AFT + RL-bonus-only runner.")


In [ ]:
# Core ML execution intentionally disabled
print("No Core ML commands are launched in this single-run RL stability notebook.")


In [ ]:
# Required RL suite intentionally disabled
print("Skipping RL required suite in this single-run stability notebook.")


In [ ]:
# Execute only the stabilized RoPE comparison run
if INCLUDE_RL_BONUS:
    stable_name = f"rl_bonus_pos_rope_stable_seed{RL_COMPARE_SEED}"
    cmd = (
        f"python src/train.py experiment.name={stable_name} "
        f"agent=td3_transformer env=hopper_hidden_vel +pos_encoding=rope seed={RL_COMPARE_SEED} "
        f"total_steps={RL_TOTAL_STEPS} eval_interval={RL_EVAL_INTERVAL} eval_episodes={RL_EVAL_EPISODES} "
        "agent.actor.obs_norm=true "
        "agent.lr=1e-4 agent.exploration_noise=0.05 "
        "agent.target_noise=0.1 agent.noise_clip=0.25 agent.tau=0.0025 "
        "logging.wandb.tags=[final,stability_compare]"
    )
    rl_bonus_commands = [with_overrides(cmd, RL_COMMON_OVERRIDES)]
    run_block("rl_bonus", RL_DIR, rl_bonus_commands)
else:
    print("Skipping stabilized RoPE comparison. Set INCLUDE_RL_BONUS=1 to enable.")


In [ ]:
# Final artifact checks and progress summary
print("Progress files:")
!ls -la /kaggle/working/SAiDL_Assignment/.kaggle_runner_progress || true

print("Core ML auto docs:")
!ls -la /kaggle/working/SAiDL_Assignment/core_ml/report/auto || true

print("RL auto docs:")
!ls -la /kaggle/working/SAiDL_Assignment/rl/report/auto || true

print("Core ML outputs (latest):")
!find /kaggle/working/SAiDL_Assignment/core_ml/outputs -maxdepth 3 -type d | tail -n 10 || true

print("RL outputs (latest):")
!find /kaggle/working/SAiDL_Assignment/rl/outputs -maxdepth 3 -type d | tail -n 10 || true